# Atari 强化学习：从单任务到持续学习

Pong 中的智能体根据画面移动球拍。如果同一个模型接着学习 Breakout，原来的球拍控制能力可能随之退化。

正文先介绍 DQN 和 PPO，再通过联合训练、顺序训练、EWC 和 GPM 比较多个游戏共享网络时的学习与遗忘。默认读者熟悉 Python、PyTorch、梯度下降和基本矩阵运算。

补充阅读：[概念与训练设定](#appendix-a) · [公式与数值例子](#appendix-b) · [方法来源](#appendix-c)。完整训练入口见 [附录 D](#appendix-d)。

先按 [本课说明](README.md) 安装依赖。下面的单元格显示当前运行环境。


In [ ]:
from biai.atari.scripts.tutorial_examples import (
    run_dqn_update_demo,
    run_ewc_penalty_demo,
    run_gae_demo,
    runtime_summary,
)

runtime_summary()

## 1. 智能体与游戏的交互

本教程使用三个游戏。下表的动图与规则说明来自 [ALE 官方文档](https://ale.farama.org/)，点击游戏名称可查看相应官方文档。

| [Pong（乒乓球）](https://ale.farama.org/environments/pong/) | [Breakout（打砖块）](https://ale.farama.org/environments/breakout/) | [Space Invaders（太空侵略者）](https://ale.farama.org/environments/space_invaders/) |
| :--- | :--- | :--- |
| <img src="https://ale.farama.org/_images/pong.gif" alt="Pong：右侧球拍由智能体控制" width="160"> | <img src="https://ale.farama.org/_images/breakout.gif" alt="Breakout：底部球拍接球，击碎上方砖块" width="160"> | <img src="https://ale.farama.org/_images/space_invaders.gif" alt="Space Invaders：底部炮台向敌人射击" width="160"> |
| **玩法**：上下移动右侧球拍，与电脑控制的左侧球拍对打。 | **玩法**：左右移动底部球拍，让球反弹并击碎上方砖块。 | **玩法**：左右移动底部炮台，射击敌人并躲避炮火。 |
| **得分**：球越过对手球拍获得正奖励，越过自己的球拍获得负奖励。 | **得分**：击碎砖块得分，分值随砖块颜色变化。 | **得分**：击毁敌人得分，后排敌人的分值更高。 |

三个游戏的计分尺度不同，后文分别比较各游戏的得分变化。

每一步，智能体观察状态 $s_t$，选择动作 $a_t$，环境返回奖励 $r_t$ 和下一状态 $s_{t+1}$。一次击球可能不会立刻得分，却会影响之后的比赛结果，因此学习需要考虑未来的累积奖励。

这里用最近四帧经过预处理的灰度图像表示状态，让网络同时获得位置和运动信息。训练时通过探索收集经验，评估时选择模型当前认为最好的动作，并记录完整游戏回合的原始得分。

训练使用的奖励与回合边界经过处理，具体约定及价值函数的定义见 [附录 A](#appendix-a)。


## 2. DQN：学习动作价值

DQN 为每个可选动作估计未来的折扣累积奖励，再据此选择动作。学习目标由当前奖励与下一状态的价值估计组成。

经验回放保存交互记录，随机抽取小批量数据更新网络；目标网络定期同步参数，减轻学习目标随更新变化的影响。训练采用 $\epsilon$-greedy 探索：以概率 $\epsilon$ 随机行动，其余时候选择价值最大的动作。

TD 目标、一次更新的数值例子和代码演示见 [附录 B.1](#appendix-dqn)。


## 3. PPO：学习动作策略

PPO 学习给定状态时各动作的选择概率，同时估计状态价值。它先收集一段交互数据，再判断哪些动作的结果好于当前预期，并据此调整动作概率。

PPO 的裁剪目标限制继续放大概率变化所能带来的收益。实际训练还包含价值损失与熵奖励，分别用于学习价值估计和鼓励探索。DQN 可以反复使用较早的回放数据，PPO 的策略更新主要使用刚收集的数据。

优势、GAE、裁剪目标与完整损失的说明见 [附录 B.2](#appendix-ppo)。


## 4. 单任务训练

先分别用 DQN 和 PPO 学习 Pong，各训练 200 万步。两者接收相同形式的画面，一个预测动作价值，另一个输出动作概率。

训练损失衡量模型与当前学习目标的差距，游戏得分反映实际表现。强化学习中的目标和探索行为都在变化，因此训练时定期运行完整回合，观察得分是否提高。

下面定义后续案例共用的参数。所有案例使用训练种子 `0`，启用确定性计算；DQN 和 PPO 都使用 8 个环境，小批量大小为 32。`--steps` 统计所有环境合计的交互次数。PPO 每个环境采集 128 步，将这一批交互数据训练 4 轮；DQN 达到学习起点后，每累计 4 条交互记录更新一次。

调用中的 `--dry-run` 只打印命令，移除后开始训练。结果写入 `results/`，已有案例可用 `--force` 重新运行；新训练及评估成功后才替换所选结果。图表和动图则读取 `assets/` 中的参考数据，自己的训练不会改变这些素材。

示例安排 10 个过程评估点，每点评估 10 个完整回合，记录原始游戏得分。Pong 约每 20 万步、Breakout 约每 5 万步评估一次，PPO 会等当前一批数据更新完再评估。下方参考图中，Pong 有 8 个实际记录点，Breakout 有 10 个；折线直接连接这些均分，反映同一次训练的变化。各案例的预算和结果见[教学结果](../../assets/README.md)。


In [ ]:
from biai.atari.scripts.run_experiments import main as run_experiment_matrix

PPO = (
    "--algorithms",
    "ppo",
    "--seed",
    "0",
    "--deterministic",
    "--num-envs",
    "8",
    "--env-backend",
    "async",
    "--compile-ppo",
)
DQN = (
    "--algorithms",
    "dqn",
    "--seed",
    "0",
    "--deterministic",
    "--num-envs",
    "8",
    "--env-backend",
    "async",
    "--compile-dqn",
)
SEQUENCE = ("--task-steps", "1000448", "500000", "500000")

run_experiment_matrix(
    (
        "train",
        "single",
        *PPO,
        "--games",
        "Pong-v5",
        "--steps",
        "2000000",
        "--eval-points",
        "10",
        "--dry-run",
    )
)
run_experiment_matrix(
    (
        "train",
        "single",
        *DQN,
        "--games",
        "Pong-v5",
        "--steps",
        "2000000",
        "--eval-points",
        "10",
        "--dry-run",
    )
)

![单任务学习曲线](../../assets/figures/single_learning.png)

图中横轴为所有环境合计的交互步数，纵轴为 10 个完整回合的平均原始得分。Pong 中，DQN 从 250k 步时的 −18.2 提高到最终 7.2；PPO 中途达到 12.9，最终为 8.9。训练更久并不保证每次评估都提高。

Breakout 各训练 50 万步。DQN 在 350k 步达到 29.7，400k 步降至 4.6，最终为 10.4；PPO 在约 350k 步为 11.2，最终为 4.2。两个算法在训练中都有明显起伏。

| 单任务 PPO：Pong，均分 8.9 | 单任务 DQN：Pong，均分 7.2 |
| :---: | :---: |
| ![PPO Pong 最终策略](../../assets/videos/single-ppo-pong-pong.gif) | ![DQN Pong 最终策略](../../assets/videos/single-dqn-pong-pong.gif) |

这些策略动图取自确定性评估录像的前 10 秒，以每秒 12 帧播放。它们便于观察动作，但只展示一个回合的片段；标题分数来自 10 个完整回合。第 1 节的官方动图用于介绍游戏，这里的动图才来自训练后的模型。[完整录像与素材来源](../../assets/README.md)。

也可以将游戏改为 `Breakout-v5`，把 `--steps` 设为 `500000`，比较两种算法在相同交互步数下的学习进展。


## 5. 联合多任务训练

Pong、Breakout 和 Space Invaders 共享卷积特征网络，各自保留独立的输出头。DQN 每个游戏有一个动作价值头，PPO 则有策略头和价值头。训练和评估时，程序都根据已知的游戏名称选择对应的头；任务设定见 [附录 A](#appendix-a)。

一个游戏的更新会改变其他输出头接收到的共享特征，既可能帮助其他游戏，也可能干扰它们。联合训练在整个过程中反复交替采样三个游戏，使每个游戏持续获得训练数据。

PPO 每次从选中的游戏收集一批交互数据（rollout），再更新参数。8 个环境各提供 128 步，共 1024 条交互记录；收集这一批数据期间，策略参数保持固定。

这里训练合计 150 万步，`--steps` 统计所有游戏的交互总量。观察时逐个比较游戏得分，因为不同游戏的奖励尺度不同。


In [ ]:
run_experiment_matrix(("train", "multitask", *PPO, "--steps", "1500000", "--dry-run"))
run_experiment_matrix(("train", "multitask", *DQN, "--steps", "1500000", "--dry-run"))

联合训练的最终评估如下。

| 算法 | 游戏 | 最终评估均分 |
|---|---|---|
| PPO | Pong-v5 | -16 |
| PPO | Breakout-v5 | 6.7 |
| PPO | SpaceInvaders-v5 | 287.5 |
| DQN | Pong-v5 | -15.5 |
| DQN | Breakout-v5 | 4.2 |
| DQN | SpaceInvaders-v5 | 337 |

| 算法 | Pong 交互步数 | Breakout 交互步数 | Space Invaders 交互步数 |
| --- | ---: | ---: | ---: |
| PPO | 514,048 | 506,880 | 479,072 |
| DQN | 502,376 | 499,560 | 498,064 |

联合模型在三个游戏间共享 150 万步预算。Pong 只获得约 50 万步，而单任务模型训练了 200 万步，所以两者的得分差距不能全部解释为任务干扰。这里展示的是联合训练结束后的评估结果。


## 6. 顺序训练与遗忘

顺序训练改变了数据的访问方式：先训练 Pong，再训练 Breakout，最后训练 Space Invaders。进入新任务后，只使用当前游戏的数据，DQN 的回放缓冲区也重新建立。

旧任务的输出头虽然不再更新，共享特征网络仍会变化。同一幅画面经过共享网络后可能得到不同的特征，旧输出头也就可能选择不同的动作，导致已学技能退化。

每完成一个阶段，都评估已经学过的游戏。把阶段放在行、游戏放在列，就得到阶段得分矩阵。沿同一列向下看，可以比较一个游戏刚学完时与后续阶段的得分，从而区分“没有学会”和“学会后遗忘”。

PPO 三个阶段分别训练 1,000,448、500,000、500,000 步，先用较长的 Pong 阶段学习需要保留的技能；DQN 每个阶段训练 500,000 步。顺序训练中的 `--steps` 指每个任务的步数，`--task-steps` 则按任务顺序分别指定。

下面的示例为每个任务阶段分别安排 10 个过程评估点，并保留阶段结束后的已学任务评估。各阶段按自身预算计算时点，不共用固定步数间隔。


In [ ]:
run_experiment_matrix(
    (
        "train",
        "continual",
        *PPO,
        "--method",
        "finetune",
        *SEQUENCE,
        "--episodes",
        "10",
        "--eval-points",
        "10",
        "--dry-run",
    )
)
run_experiment_matrix(
    (
        "train",
        "continual",
        *DQN,
        "--method",
        "finetune",
        "--steps",
        "500000",
        "--episodes",
        "10",
        "--eval-points",
        "10",
        "--dry-run",
    )
)

## 7. EWC：限制重要参数的变化

EWC 在旧任务结束时保存参数及其重要性。学习新任务时，如果重要参数偏离原值，就增加相应的损失惩罚。

这些参数也可能参与新任务学习，因此加重惩罚可能同时限制新任务所需的变化。下面在顺序 PPO 中加入 EWC，观察旧任务保留和新任务学习。惩罚公式、重要性估计与数值例子见 [附录 B.3](#appendix-ewc)。


In [ ]:
run_experiment_matrix(
    (
        "train",
        "continual",
        *PPO,
        "--method",
        "ewc",
        *SEQUENCE,
        "--ewc-lambda",
        "0.4",
        "--episodes",
        "10",
        "--eval-points",
        "10",
        "--dry-run",
    )
)

## 8. GPM：限制更新方向

GPM 在每个任务结束后，从网络特征中提取主要输入方向并保存下来。学习新任务时，去掉参数更新中沿这些方向的分量，以减少对旧任务表示的改变。

保护的方向越多，留给新任务的更新方向就越少。下面比较加入 GPM 后的阶段得分；投影公式与二维例子见 [附录 B.4](#appendix-gpm)，Adam 更新的处理见 [附录 C](#appendix-c)。


In [ ]:
run_experiment_matrix(
    (
        "train",
        "continual",
        *PPO,
        "--method",
        "gpm",
        *SEQUENCE,
        "--episodes",
        "10",
        "--eval-points",
        "10",
        "--dry-run",
    )
)
run_experiment_matrix(
    (
        "train",
        "continual",
        *DQN,
        "--method",
        "gpm",
        "--steps",
        "500000",
        "--episodes",
        "10",
        "--eval-points",
        "10",
        "--dry-run",
    )
)

## 9. 阶段评估：同时看保留与新任务学习

下面比较三种 PPO 顺序训练方法。它们分别从头训练，在 Pong 阶段结束时得到相同的模型。图中 Sequential 表示普通顺序训练；每个面板对应一个游戏，横轴表示刚完成的训练阶段。

![PPO 三任务阶段得分](../../assets/figures/ppo_continual.png)

尚未学习的游戏留空，因此 Space Invaders 只有最后阶段的一个点。每点是 10 个完整回合的均分；三个游戏使用各自的奖励坐标，不合并计算总分。

先看 Pong。三种方法刚学完时均为 12.6，普通顺序最终降到 −19.9；GPM 经过后续两个任务后仍为 11.8，保留了较多旧任务表现。EWC 的轨迹为 12.6 → −19.7 → −12.9，先明显退化，最后部分恢复。

再看 Breakout。普通顺序从 7.6 降到 0.5；EWC 为 7.8 → 9.8，GPM 为 6.0 → 9.4。最后的新任务 Space Invaders 分别为 242.0、334.5、262.5。EWC 的 Space Invaders 得分较高，GPM 则保留了更多 Pong 能力：这两个指标反映了不同方面的表现。

下面先展示三种方法的共同起点，再展示完成全部任务后的 Pong 策略。

| 共同起点：学完 Pong，均分 12.6 |
| :---: |
| ![PPO 共同首阶段策略](../../assets/videos/continual-ppo-finetune-pong-stage1.gif) |

| 普通顺序：−19.9 | EWC：−12.9 | GPM：11.8 |
| :---: | :---: | :---: |
| ![PPO 普通顺序最终策略](../../assets/videos/continual-ppo-finetune-pong.gif) | ![PPO EWC 最终策略](../../assets/videos/continual-ppo-ewc-pong.gif) | ![PPO GPM 最终策略](../../assets/videos/continual-ppo-gpm-pong.gif) |

这组结果来自训练种子 `0`。若想了解差异是否稳定，可以用其他种子重复训练。完整阶段矩阵和每回合得分见[教学结果](../../assets/README.md)。


### 独立评估

训练完成后，可以加载保存的模型重新评估。下面预览 GPM 的 DQN、PPO 评估命令，在每个游戏中各运行 10 个完整回合。`--results-dir` 可选择结果根目录，已有评估可用 `--force` 替换；将 `--method` 改为 `finetune` 或 `ewc`，可评估对应方法保存的模型。


In [ ]:
run_experiment_matrix(
    (
        "evaluate",
        "continual",
        "--algorithms",
        "dqn",
        "ppo",
        "--method",
        "gpm",
        "--episodes",
        "10",
        "--deterministic",
        "--dry-run",
    )
)

## 10. 扩展：DQN 中的 EWC

DQN 使用逐样本 TD 损失梯度的平方均值估计参数重要性，再加入 EWC 惩罚。它与 PPO 中策略经验 Fisher 的含义不同，说明见 [附录 B.3](#appendix-ewc)。

可以比较普通 DQN、DQN + EWC 和 DQN + GPM 的阶段得分，先看各游戏刚学完时的表现，再观察后续变化。


In [ ]:
run_experiment_matrix(
    (
        "train",
        "continual",
        *DQN,
        "--method",
        "ewc",
        "--steps",
        "500000",
        "--episodes",
        "10",
        "--eval-points",
        "10",
        "--dry-run",
    )
)

![DQN 三任务阶段得分](../../assets/figures/dqn_continual.png)

DQN 的 Pong 首阶段训练 50 万步，三种方法刚学完时均为 −15.0。相比之下，前面的单任务模型训练了 200 万步，最终为 7.2。这组顺序实验在第一个阶段就尚未充分学会 Pong，因此解读后续低分时，也要考虑这个较弱的起点。

Breakout 更清楚地展示了差异：普通顺序从 22.0 降到 3.7，EWC 从 22.0 降到 0.1，这次运行中，EWC 没能保留 Breakout 的表现。GPM 从 5.6 降到 1.7，下降较小，但它最初学到的水平也更低。比较方法时应同时看新任务学到了多少、旧任务后来还剩多少。


## 11. 在相同交互预算下比较单任务、联合和顺序训练

前面的 Pong 单任务、联合和顺序实验使用了不同的训练步数，得分差距同时受到训练方式与交互量的影响。下面让每个游戏获得相同的交互次数，再比较三种训练方式。

三个游戏分别训练单任务模型，再训练一个联合模型，以及普通顺序、EWC、GPM 三种顺序模型。DQN 和 PPO 各有七个案例，共十四个。每个游戏的主训练循环固定采集 50 万次交互：单任务每个模型训练 50 万步，联合与顺序模型合计各训练 150 万步。

联合训练随机选择尚有剩余步数的游戏；PPO 最后一批采样按剩余步数截短，避免超过预算。`--steps` 可统一改变每游戏的交互次数。各方法的参数更新、回放及重要性估计仍有不同的计算开销，相同交互次数并不等于相同训练时间。

下面分别预览短训练和完整实验，移除相应调用中的 `--dry-run` 后执行。短训练用于检查流程，结果写入 `results/atari-matched-smoke/`；完整实验写入 `results/atari-matched/`。这些是自己的运行结果，与前面的参考图表分开保存。


In [ ]:
from IPython.display import Markdown, display

from biai.atari.scripts.tutorial_examples import matched_budget_results, teaching_results
from biai.paths import RESULTS_DIR

matched_results_dir = RESULTS_DIR / "atari-matched"
matched_smoke_dir = RESULTS_DIR / "atari-matched-smoke"
run_experiment_matrix(
    (
        "train",
        "teaching",
        "--matched-budget",
        "--smoke",
        "--results-dir",
        str(matched_smoke_dir),
        "--dry-run",
    )
)
run_experiment_matrix(
    (
        "train",
        "teaching",
        "--matched-budget",
        "--results-dir",
        str(matched_results_dir),
        "--dry-run",
    )
)
display(Markdown(matched_budget_results(matched_results_dir)))

### 查看阶段得分

完整实验完成后，下面读取三种顺序训练方法的阶段得分。先比较每个游戏刚学完时的分数，再沿同一游戏查看后续变化，就能看出相同交互预算下的新任务学习和旧任务保持情况。


In [ ]:
if all(
    (
        matched_results_dir / f"continual-{algorithm}-{method}" / "evaluation/evaluation.json"
    ).is_file()
    for algorithm in ("dqn", "ppo")
    for method in ("finetune", "ewc", "gpm")
):
    display(
        Markdown(
            "\n\n".join(
                teaching_results(method, matched_results_dir)
                for method in ("finetune", "ewc", "gpm")
            )
        )
    )

### 可选扩展

- 可以调整共享卷积层的深度，或改变 EWC/GPM 的保护强度，每次固定其余条件，观察各游戏的新任务学习与旧任务保持。
- 如果对元学习感兴趣，可以探索跨游戏的初始化与适应。为此需要定义训练任务的采样方式，以及新游戏上的适应评估，才能考察模型是否学到了便于适应的起点。


<a id="appendix-a"></a>

## 附录 A：概念与训练设定

### 回报、价值与优势

即时奖励 $r_t$ 来自一次交互。折扣回报 $G_t$ 累加当前及未来的奖励：

$$G_t=\sum_{k=0}^{\infty}\gamma^k r_{t+k}=r_t+\gamma G_{t+1}.$$

$0\leq\gamma<1$ 是折扣因子。训练希望提高期望折扣回报，但一次轨迹只给出其中一个样本。

给定策略 $\pi$，状态价值和动作价值分别为

$$V^\pi(s)=\mathbb E_\pi[G_t\mid s_t=s],\qquad Q^\pi(s,a)=\mathbb E_\pi[G_t\mid s_t=s,a_t=a].$$

前者对策略可能选择的动作取平均，后者先固定当前动作，再按策略继续行动。二者之差就是优势：

$$A^\pi(s,a)=Q^\pi(s,a)-V^\pi(s).$$

因此，正优势表示这个动作好于该状态下的平均选择。用下一状态的价值估计代替尚未观察到的未来回报，称为自举（bootstrapping）；DQN 和 PPO 都会用到它。

### Atari 中的状态、奖励与回合

图像经过缩放和灰度处理，四帧堆叠后的形状是 $(4,84,84)$。动作默认重复 4 个模拟器帧，脚本中的步数按智能体与环境的交互次数统计。

| 约定 | 训练 | 评估 |
| --- | --- | --- |
| 奖励 | 每步奖励按符号变成 −1、0 或 1 | 累加原始游戏奖励 |
| 回合边界 | 掉命可作为训练终止信号 | 运行完整游戏回合 |
| 动作选择 | DQN 使用随机探索，PPO 从策略中采样 | DQN 取最大动作价值，PPO 取最大动作概率 |

所以，训练中的终止信号不一定表示整局游戏结束。若只是时间限制截断，仍可用截断前最后状态的价值自举，但优势递推必须在此结束，不能接到重置后的新回合。环境处理见 [atari_env.py](environments/atari_env.py)。

### 持续学习中的可用信息

本教程采用任务身份已知的多头设定：游戏名称决定使用哪个输出头，任务切换时间也由训练程序给定。模型学习共享特征和各任务的行为，无需自行识别正在运行的游戏。

联合训练反复访问所有游戏；顺序训练进入新阶段后，训练更新只使用当前游戏的数据。EWC 保留旧参数与重要性，GPM 保留输入子空间。阶段评估会重新运行旧游戏，但评估数据不参与训练更新。相关设定的区分见 [Three scenarios for continual learning](https://arxiv.org/abs/1904.07734)。


<a id="appendix-b"></a>

## 附录 B：公式与数值例子

这里用小规模数值说明更新机制。代码演示复用开头导入的函数，运行首个代码单元后即可单独执行。

<a id="appendix-dqn"></a>

### B.1 DQN：一次 TD 更新

DQN 用下一状态的最大动作价值构造目标，用来估计采取最优后续动作时的回报：

$$y_t=r_t+\gamma(1-d_t)\max_a Q_{\mathrm{target}}(s_{t+1},a),\qquad \mathcal L_{\mathrm{DQN}}=\mathbb E[(Q(s_t,a_t)-y_t)^2].$$

$d_t=1$ 表示训练终止；时间限制截断不应直接当作这种终止。目标网络不参与这一步的梯度计算，而是定期从预测网络同步参数。

取 $r_t=1$、$\gamma=0.9$、下一状态的最大动作价值为 2，当前预测 $Q(s_t,a_t)=0.5$：

| 交互结束时的状态 | 目标 $y_t$ | TD 误差 $y_t-Q(s_t,a_t)$ | 平方损失 |
| --- | --- | --- | --- |
| 未终止，$d_t=0$ | 2.8 | 2.3 | 5.29 |
| 训练终止，$d_t=1$ | 1.0 | 0.5 | 0.25 |

在未终止的一行，损失对预测值的导数为 $2(0.5-2.8)=-4.6$，梯度下降会推动这个预测值上升。网络使用共享参数，更新也可能影响其他动作的价值。

下面的函数用合成图像批次执行仓库中的 DQN 更新。可对照上面的目标公式，查看返回的张量形状与 TD 损失。


In [ ]:
run_dqn_update_demo()

<a id="appendix-ppo"></a>

### B.2 PPO：优势、裁剪与完整损失

先考虑回合内部的相邻时间步。时序差分误差与 GAE 递推为

$$\delta_t=r_t+\gamma V(s_{t+1})-V(s_t),\qquad \hat A_t=\delta_t+\gamma\lambda\hat A_{t+1}.$$

训练终止时，下一状态的价值取零；截断时保留最后状态的价值，但不跨回合累加优势。$\lambda$ 控制后续 TD 误差的权重。价值网络的回报目标为 $\hat R_t=\hat A_t+V_{\mathrm{old}}(s_t)$。

`run_gae_demo()` 使用三步轨迹，奖励为 $(0,0,1)$，价值估计为 $(0.2,0.3,0.4)$，最后一步终止；取 $\gamma=0.99$、$\lambda=0.95$，得到：

| 时间步 | 奖励 | 价值估计 | TD 误差 | 优势估计 | 回报目标 |
| --- | --- | --- | --- | --- | --- |
| 0 | 0 | 0.2 | 0.097 | 0.7180 | 0.9180 |
| 1 | 0 | 0.3 | 0.096 | 0.6603 | 0.9603 |
| 2 | 1 | 0.4 | 0.600 | 0.6000 | 1.0000 |

例如，最后一步优势为 $1-0.4=0.6$，前一步为 $0.096+0.99\times0.95\times0.6=0.6603$。

策略更新使用新旧动作概率之比

$$\rho_t(\theta)=\frac{\pi_\theta(a_t\mid s_t)}{\pi_{\mathrm{old}}(a_t\mid s_t)},$$

并最大化裁剪目标

$$L_{\mathrm{clip}}=\mathbb E\left[\min\left(\rho_t\hat A_t,\operatorname{clip}(\rho_t,1-\epsilon,1+\epsilon)\hat A_t\right)\right].$$

取 $\epsilon=0.2$，比较单个样本的目标值：

| 优势 $\hat A_t$ | 概率比 $\rho_t$ | 未裁剪项 | 裁剪项 | 取两者较小值 |
| --- | --- | --- | --- | --- |
| +1 | 1.5 | 1.5 | 1.2 | 1.2 |
| +1 | 0.5 | 0.5 | 0.8 | 0.5 |
| −1 | 0.5 | −0.5 | −0.8 | −0.8 |
| −1 | 1.5 | −1.5 | −1.2 | −1.5 |

正优势动作的概率提高到一定程度后，继续提高不再增加这项收益；负优势动作的概率下降也有对应限制。概率比本身没有被强制限制在区间内，朝不利方向的变化仍会降低目标值。

代码最小化的完整损失为

$$\mathcal L_{\mathrm{PPO}}=-L_{\mathrm{clip}}+c_V L_V-c_H\mathcal H(\pi).$$

$L_V$ 用回报目标训练价值网络，$\mathcal H$ 是策略熵，$c_V$ 和 $c_H$ 控制两项的权重。负号使最小化损失对应最大化策略目标与熵。当前实现还对价值变化进行裁剪，取裁剪前后平方误差的较大者，见 [ppo_minibatch_loss](algorithms/ppo.py)。

下面可以核对三步轨迹的 GAE 结果。


In [ ]:
run_gae_demo()

<a id="appendix-ewc"></a>

### B.3 EWC：相同位移，不同惩罚

对一个旧任务，EWC 的惩罚和新任务损失相加：

$$\mathcal L=\mathcal L_{\mathrm{new}}+\frac{\lambda_{\mathrm{EWC}}}{2}\sum_i I_i(\theta_i-\theta_i^*)^2.$$

$\theta_i^*$ 是旧任务结束时的参数，$I_i$ 表示重要性。多个旧任务的惩罚相加。取两个参数的重要性为 $(100,1)$，$\lambda_{\mathrm{EWC}}=1$，分别只移动一个参数：

| 相对旧参数的位移 | EWC 惩罚 | 等权惩罚（两个 $I_i$ 都为 1） |
| --- | --- | --- |
| $(0.1,0)$ | 0.5 | 0.005 |
| $(0,0.1)$ | 0.005 | 0.005 |

两次位移大小相同，EWC 对第一个参数的惩罚是第二个的 100 倍，因此更新会更受限制。

本教程中，PPO 对旧任务样本逐个计算策略负对数似然的梯度，再取平方均值：

$$I_i\approx\frac{1}{N}\sum_{n=1}^{N}\left[\frac{\partial[-\log\pi_\theta(a_n\mid s_n)]}{\partial\theta_i}\right]_{\theta=\theta^*}^{2}.$$

这是策略的对角经验 Fisher 估计，覆盖共享网络与策略头，不直接约束价值头。DQN 则把其中的样本损失换为 TD 均方损失；它衡量 TD 损失对参数的敏感程度，是本教程采用的重要性近似，不是策略经验 Fisher。

下面调用 DQN 上的 EWC 实现，比较参考权重处与移动权重后的惩罚。


In [ ]:
run_ewc_penalty_demo()

<a id="appendix-gpm"></a>

### B.4 GPM：一个二维投影

对固定输入 $x$，线性层 $y=Wx$ 的输出变化为 $\Delta y=\Delta W x$。设 $U$ 的列向量是旧任务输入子空间的一组标准正交基，即 $U^\top U=I$，则

$$\Delta W=\Delta W_{\mathrm{raw}}(I-UU^\top),\qquad \Delta W U=0.$$

$\Delta W_{\mathrm{raw}}$ 是投影前的更新，$I$ 是单位矩阵。若旧输入位于 $U$ 张成的子空间中，投影后这次更新不会改变该输入对应的线性输出。

例如，旧输入沿第一坐标方向，取 $U=(1,0)^\top$，原始更新为 $\Delta W_{\mathrm{raw}}=(2,3)$：

$$I-UU^\top=\begin{pmatrix}0&0\\0&1\end{pmatrix},\qquad\Delta W=(0,3).$$

| 输入 | 投影前的输出变化 | 投影后的输出变化 |
| --- | --- | --- |
| 旧方向 $(1,0)^\top$ | 2 | 0 |
| 另一方向 $(0,1)^\top$ | 3 | 3 |

如果两个坐标方向都受到保护，整个二维空间都被占用，允许的更新就变为零。这说明保护范围增大为何可能限制新任务学习。

实现中按各层输入的能量选择主要方向，新任务只向已有子空间补充未覆盖的部分。输入子空间由有限样本估计，前面各层的特征也会变化，因此单层固定输入下的结论不能保证整个旧策略不变。卷积层使用感受野中的输入块；本实现还把偏置视为常数输入的权重一并处理。


<a id="appendix-c"></a>

## 附录 C：方法来源与本仓库实现

下表列出各部分的原始文献及其与本教程的对应关系。

| 内容 | 文献 | 本教程采用的部分 |
| --- | --- | --- |
| DQN | Mnih 等，[Human-level control through deep reinforcement learning](https://www.nature.com/articles/nature14236)，2015 | 从图像学习动作价值，使用经验回放与目标网络 |
| PPO | Schulman 等，[Proximal Policy Optimization Algorithms](https://arxiv.org/abs/1707.06347)，2017 | 裁剪策略目标，以及价值损失与熵奖励 |
| GAE | Schulman 等，[High-Dimensional Continuous Control Using Generalized Advantage Estimation](https://arxiv.org/abs/1506.02438) | 用多个时间步的 TD 误差估计优势 |
| EWC | Kirkpatrick 等，[Overcoming catastrophic forgetting in neural networks](https://arxiv.org/abs/1612.00796)，2017 | 对偏离旧参数的变化施加重要性加权惩罚 |
| GPM | Saha 等，[Gradient Projection Memory for Continual Learning](https://arxiv.org/abs/2103.09762)，2021 | 保存输入表示的主要子空间，约束后续更新方向 |

EWC 的惩罚形式相同，但重要性估计需要结合具体学习目标。本教程的 PPO 使用策略经验 Fisher，DQN 使用 TD 损失平方梯度近似；对应实现见 [ewc.py](algorithms/ewc.py)。

GPM 原论文在图像分类任务上通过梯度投影减少遗忘。本教程将其用于 Atari，并投影 Adam 给出的实际参数位移：Adam 的动量和逐坐标缩放会改变更新方向，因此仅投影原始梯度不足以保证实际位移满足约束。投影作用于共享网络，包含偏置，见 [subspace_projection.py](algorithms/subspace_projection.py)。


<a id="appendix-d"></a>

## 附录 D：运行全部案例

`teaching` 入口汇总全部正式案例，依次执行训练和最终评估。下面仍只预览命令。若要保留已有模型和日志，可以为新训练指定另一个结果目录。各案例的训练步数与已有结果见 [结果记录](../../assets/README.md)，产物保存位置见 [本课说明](README.md)。

阅读实现时，可以从 [DQN](algorithms/dqn.py) 和 [PPO](algorithms/ppo.py) 开始，再看 [顺序训练入口](scripts/train_continual.py) 中的任务切换与阶段评估。


In [ ]:
run_experiment_matrix(("train", "teaching", "--seed", "0", "--dry-run"))